In [ ]:
import numpy as np
from scipy.interpolate import RegularGridInterpolator
from skimage.io import imread
from edt import edt
import pandas as pd
from pathlib import Path

from calmutils.imageio.nd2_helpers import get_pixel_size

In [ ]:
in_path = '/Volumes/nn/Julia Vogtmann/Microscopy/26AM01-01_2'

mask_outer_subdirectory = 'segmentation_nuclei_edgesnap'
mask_inner_subdirectory = 'segmentation_nucleoli_edgesnap'
spot_detection_subdirectory = 'spot-detection'

out_subdirectory = 'spot-mask-distances-nested'

# whether to try to load pixel sizes from nd2 image file in coords table
# if this is False, pixel_size_default will be used instead
try_load_pixelsize = True
pixel_size_default = [1, 1, 1]

outer_id = 'nucleus'
inner_id = 'nucleolus'

coordinate_columns = ['z', 'y', 'x']
image_file_column = 'image_file'

In [ ]:
mask_files_outer = sorted((Path(in_path) / mask_outer_subdirectory).glob('[!.]*.tif'))
mask_files_inner = sorted((Path(in_path) / mask_inner_subdirectory).glob('[!.]*.tif'))
coords_table_files = sorted((Path(in_path) / spot_detection_subdirectory).glob('[!.]*.csv'))

mask_files_outer, mask_files_inner, coords_table_files

In [ ]:
out_path = Path(in_path) / out_subdirectory
if not out_path.exists():
    out_path.mkdir()

for mask_file_outer, mask_file_inner, coords_table_file in zip(mask_files_outer, mask_files_inner, coords_table_files):

    # load masks and table
    mask_outer = imread(mask_file_outer)
    mask_inner = imread(mask_file_inner)
    df_coords = pd.read_csv(coords_table_file)

    # TODO handle relative file paths?
    # get pixel size from one of the files in table
    # (we assume it is the same, which is usually true bc. we have one image per results table)
    pixel_size = pixel_size_default
    if try_load_pixelsize:
        pixel_size = get_pixel_size(next(iter(df_coords[image_file_column].unique())))
    
    # EDT from border of outer (e.g. nucleus) -> inside dist. from border
    edt_outer = edt(mask_outer, anisotropy=pixel_size)
    # EDT from inverse of inner (e.g. nucleoli) -> distance to nearest 
    edt_inner_inv = edt(mask_inner == 0, anisotropy=pixel_size)
    
    # interpolators for non-integer coords
    interp_dt_outer = RegularGridInterpolator(tuple((np.arange(s) for s in edt_outer.shape)), edt_outer, bounds_error=False, fill_value=None)
    interp_dt_inner_inv = RegularGridInterpolator(tuple((np.arange(s) for s in edt_inner_inv.shape)), edt_inner_inv, bounds_error=False, fill_value=None)
    
    # mask interpolator: use nearest neighbor
    interp_mask = RegularGridInterpolator(tuple((np.arange(s) for s in mask_outer.shape)), mask_outer, method='nearest', bounds_error=False, fill_value=None)

    coords = df_coords[coordinate_columns]

    # get distances and labels at coordinates
    ds_outer = interp_dt_outer(coords)
    ds_inner = interp_dt_inner_inv(coords)
    labels = interp_mask(coords)
    
    # sorted list of distances to border of outer (nucleus) per label (excluding bg==0)
    edt_outer_sorted_bylabel = {label: np.sort(edt_outer[mask_outer==label]) for label in np.unique(labels) if label != 0}
    
    # same for distances to inner (e.g. nucleolus) for each outer
    # NOTE: pixels within inner are ignored here
    edt_inner_inv_sorted_bylabel = {label: np.sort(edt_inner_inv[(mask_outer==label) & (mask_inner == 0)]) for label in np.unique(labels) if label != 0}
    
    # index of binary search in sorted distances lists / num pixels -> distance quantiles
    q_outer = [np.searchsorted(edt_outer_sorted_bylabel[label], d) / edt_outer_sorted_bylabel[label].size if label != 0 else np.nan 
        for d,label in zip(ds_outer, labels)]
    q_inner = [np.searchsorted(edt_inner_inv_sorted_bylabel[label], d) / edt_inner_inv_sorted_bylabel[label].size if label != 0 else np.nan
        for d,label in zip(ds_inner, labels)]


    ### ADD results as new colums
    df_coords[f'label_{outer_id}'] = labels.astype(int)

    df_coords[f'd_{outer_id}'] = ds_outer
    df_coords[f'd_{inner_id}'] = ds_inner
    
    df_coords[f'q_{outer_id}'] = q_outer
    df_coords[f'q_{inner_id}'] = q_inner

    out_file = out_path / (coords_table_file.stem + '_mask-distances.csv')
    df_coords.to_csv(out_file, index=None)

    print(f'wrote results to {out_file}.')